# Meta-Judge cho đánh giá sinh ngôn ngữ tiếng Việt

**Quy trình thực nghiệm.** Sử dụng nguồn, tham chiếu, bản dịch A/B và điểm
người chấm từ kết quả dịch đã lưu; xây dựng dữ liệu suy giảm ngữ nghĩa, đánh giá
metric và đối chiếu thứ hạng metric với đánh giá của con người.

## 1. Thiết lập thí nghiệm

Notebook tự tìm repo `work` khi chạy local, hoặc clone repo này khi chạy trên
Colab. Source, resource và notebook cùng một revision Git nên không cần upload
hay giải nén thủ công. Mọi kết quả nằm trong `ROOT/output/RUN_NAME`.

**Hai chế độ dữ liệu mặc định đều dùng lại artifact trong Git:**

| Tùy chọn | `False` (mặc định) | `True` |
|---|---|---|
| `REGENERATE_TRANSLATIONS` | Dùng `resources/source.csv` | Sinh lại hệ A bằng Gemini và hệ B bằng Google Translate |
| `REGENERATE_DAMAGE` | Dùng `resources/zero_shot.jsonl` và `few_shot.jsonl` | Sinh/resume damage mới vào output của `RUN_NAME` |

Khi bật sinh lại, các bước phía sau tự đọc artifact vừa sinh. Điểm human trong
`scores.xlsx` chỉ gắn với A/B đã chốt trong repo, nên tự tắt khi A/B được sinh
lại. Không có Gemini API nào được gọi nếu cả hai tùy chọn vẫn là `False`.

| Thư mục output | Nội dung |
|---|---|
| `translations/` | Bộ mẫu và checkpoint A/B khi chạy lại bước dịch |
| `data/` | Dữ liệu bản dịch đã kiểm tra và tập tham chiếu thí nghiệm |
| `generation/` | Zero-shot, few-shot, B1 và kiểm tra damage |
| `metrics/` | Điểm metric, tương quan và bảng kết quả |
| `analysis/` | B2, xuất điểm người chấm, phân tích lỗi và demo |
| `logs/` | Log riêng cho từng lượt chạy |

In [ ]:
import os
from pathlib import Path
import subprocess

# ── TÙY CHỌN NGƯỜI CHẠY ────────────────────────────────────────────────────
# False: dùng bản dịch A/B có sẵn trong Git. True: sinh/resume A/B mới.
REGENERATE_TRANSLATIONS = False

# False: dùng zero/few-shot damage có sẵn trong Git. True: sinh/resume damage mới.
REGENERATE_DAMAGE = False

# Key quota thường: toàn bộ mảng dùng chung rule cũ, tối đa một request mỗi 4.2s.
STANDARD_GEMINI_API_KEYS = []

# Key thuộc project có quota cao: chạy nhanh hơn theo cấu hình HIGH_QUOTA bên dưới.
# Không có API key nào thật sự "unlimited"; chỉ đưa key vào đây sau khi kiểm tra
# RPM/RPD của đúng model trên Google AI Studio. Nhiều key cùng một project vẫn
# dùng chung quota project. Không lặp key giữa hai mảng.
HIGH_QUOTA_GEMINI_API_KEYS = []

# True: cài dependency còn thiếu. Nên giữ True trên một runtime Colab mới.
INSTALL_DEPENDENCIES = True

# True: bật BERTScore, COMET và BLEURT; cần GPU để chạy nhanh.
RUN_HEAVY_METRICS = True

# True: chấm thử một config mỗi họ metric trên 24 hàng trước khi chạy full.
RUN_METRIC_SMOKE_TEST = True

# False: dừng sau smoke test. Đổi True khi smoke đã pass để chấm toàn bộ.
RUN_FULL_METRICS = False

# True: tính thêm BLEU/chrF sau khi tách từ bằng underthesea.
RUN_UNDERTHESEA = True

# ── HẰNG SỐ VÀ GIỚI HẠN THỰC NGHIỆM ─────────────────────────────────────────
# Đổi RUN_NAME khi thay model, prompt hoặc input; giữ nguyên để resume checkpoint.
RUN_NAME = "full-gemini-3-5-flash-lite-01"
GEMINI_MODEL = "gemini-3.5-flash-lite"
EXPECTED_ROWS = 300
LIMIT = None  # None dùng đủ 300 câu; số nguyên dương dùng cho pilot.
SEED = 42
MAX_OUTPUT_TOKENS = 400

# Pool thường giữ đúng rule cũ và dùng chung quota giữa các key trong mảng.
STANDARD_API_WORKERS = 1
STANDARD_API_MIN_INTERVAL_SECONDS = 4.2

# Mỗi key quota cao có pool riêng: mặc định 4 request đồng thời, cách nhau 0.25s.
# Giảm workers hoặc tăng interval nếu dashboard báo 429 do RPM.
HIGH_QUOTA_WORKERS_PER_KEY = 4
HIGH_QUOTA_MIN_INTERVAL_SECONDS = 0.25

CPU_WORKERS = 2
GPU_BATCH_SIZE = 16
HF_TOKEN = ""
LOCAL_SOURCE_VI = None
LOCAL_SOURCE_ZH = None
MANUAL_AUDIT_FILE = None
DEMO_SENTENCE = "Việt Nam đang đẩy mạnh chuyển đổi số trong giáo dục đại học."

# Repo mặc định chứa source/resource. Có thể override bằng biến môi trường.
WORK_REPO_URL = os.environ.get(
    "META_JUDGE_WORK_REPO",
    "https://github.com/thanhnghi-do-2k3/llm-as-judge.git",
)
WORK_REPO_BRANCH = "main"


def normalize_key_array(name, values):
    if not isinstance(values, (list, tuple)) or any(
        not isinstance(value, str) for value in values
    ):
        raise ValueError(f"{name} phải là mảng các chuỗi.")
    return list(dict.fromkeys(value.strip() for value in values if value.strip()))


STANDARD_GEMINI_API_KEYS = normalize_key_array(
    "STANDARD_GEMINI_API_KEYS", STANDARD_GEMINI_API_KEYS
)
HIGH_QUOTA_GEMINI_API_KEYS = normalize_key_array(
    "HIGH_QUOTA_GEMINI_API_KEYS", HIGH_QUOTA_GEMINI_API_KEYS
)
overlapping_keys = set(STANDARD_GEMINI_API_KEYS) & set(HIGH_QUOTA_GEMINI_API_KEYS)
if overlapping_keys:
    raise ValueError("Một Gemini key không được nằm trong cả hai mảng.")
ALL_GEMINI_API_KEYS = STANDARD_GEMINI_API_KEYS + HIGH_QUOTA_GEMINI_API_KEYS

if (REGENERATE_TRANSLATIONS or REGENERATE_DAMAGE) and not ALL_GEMINI_API_KEYS:
    raise ValueError(
        "Bật sinh lại nhưng chưa điền STANDARD_GEMINI_API_KEYS hoặc "
        "HIGH_QUOTA_GEMINI_API_KEYS."
    )


def is_work_repo(path):
    path = Path(path)
    return (
        (path / "resources" / "source.csv").is_file()
        and (path / "resources" / "scores.xlsx").is_file()
        and (path / "meta-judge" / "src" / "vn_meta_judge").is_dir()
    )


cwd = Path.cwd().resolve()
local_candidates = [cwd, cwd / "work"]
ROOT = next((path for path in local_candidates if is_work_repo(path)), None)

if ROOT is None:
    clone_root = Path("/content/llm-as-judge").resolve()
    if not is_work_repo(clone_root):
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                WORK_REPO_BRANCH,
                WORK_REPO_URL,
                str(clone_root),
            ],
            check=True,
        )
    ROOT = clone_root

ROOT = Path(ROOT).resolve()
RESOURCES_DIR = ROOT / "resources"
CODE_DIR = ROOT / "meta-judge"

REPO_TRANSLATIONS_FILE = RESOURCES_DIR / "source.csv"
REPO_SCORES_FILE = RESOURCES_DIR / "scores.xlsx"
REPO_DAMAGE_FILES = {
    "zero_shot": RESOURCES_DIR / "zero_shot.jsonl",
    "few_shot": RESOURCES_DIR / "few_shot.jsonl",
}

# Các biến ACTIVE luôn trỏ tới artifact mà pipeline phía dưới phải sử dụng.
ACTIVE_TRANSLATIONS_FILE = REPO_TRANSLATIONS_FILE
ACTIVE_SOURCE_FILE = None  # source.csv đã chứa nguồn và tham chiếu.
ACTIVE_SCORES_FILE = REPO_SCORES_FILE

### Môi trường thực thi

Giữ nguyên scientific stack NumPy/Pandas/SciPy có sẵn của runtime, sau đó cô
lập BERTScore, COMET và BLEURT theo từng thư mục package. Probe import dừng sớm
nếu cache cài dở; không tạo virtualenv và không thay PyTorch của Colab.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys

ROOT = Path(ROOT).expanduser().resolve()
CODE_DIR = Path(CODE_DIR).expanduser().resolve()

if not (CODE_DIR / "src" / "vn_meta_judge" / "notebook_workflow.py").is_file():
    raise FileNotFoundError(f"Source Git chưa đầy đủ: {CODE_DIR}")
required_inputs = []
if not REGENERATE_TRANSLATIONS:
    required_inputs.extend([REPO_TRANSLATIONS_FILE, REPO_SCORES_FILE])
if not REGENERATE_DAMAGE:
    required_inputs.extend(REPO_DAMAGE_FILES.values())
for required_input in required_inputs:
    if not Path(required_input).is_file():
        raise FileNotFoundError(f"Thiếu resource trong Git: {required_input}")

if sys.version_info >= (3, 13) and RUN_HEAVY_METRICS:
    raise RuntimeError("Metric nặng cần Python 3.12 trở xuống.")

os.environ["HF_HOME"] = str(ROOT / "cache" / "huggingface")
os.environ["NLTK_DATA"] = str(ROOT / "cache" / "nltk")
os.environ["TORCH_HOME"] = str(ROOT / "cache" / "torch")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MPLBACKEND"] = "Agg"

METRIC_PYTHON = Path(sys.executable).resolve()
HEAVY_WORKER_PATHS = {}


def run_checked(command, label, env=None):
    result = subprocess.run(
        command,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        tail = "\n".join(result.stdout.splitlines()[-40:])
        raise RuntimeError(f"{label} thất bại:\n{tail}")
    return result.stdout


SCIENTIFIC_DISTRIBUTIONS = ("numpy", "pandas", "scipy")
VERSION_SCRIPT = """
import importlib.metadata as metadata
import json
import sys

versions = {}
for name in sys.argv[1:]:
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        pass
print(json.dumps(versions, sort_keys=True))
"""


def installed_versions(names):
    output = run_checked(
        [sys.executable, "-c", VERSION_SCRIPT, *names],
        "Đọc phiên bản scientific stack",
    )
    return json.loads(output.strip())


def ensure_kernel_matches_disk(installed):
    mismatches = []
    for name in SCIENTIFIC_DISTRIBUTIONS:
        module = sys.modules.get(name)
        loaded = getattr(module, "__version__", None)
        current = installed.get(name)
        if loaded and current and loaded != current:
            mismatches.append(f"{name}: RAM={loaded}, disk={current}")
    if mismatches:
        raise RuntimeError(
            "Runtime đã nạp scientific stack khác phiên bản đang có trên disk: "
            + "; ".join(mismatches)
            + ". Hãy chọn Runtime → Disconnect and delete runtime, kết nối lại "
            "rồi Run all notebook mới nhất."
        )


scientific_versions = installed_versions(SCIENTIFIC_DISTRIBUTIONS)
ensure_kernel_matches_disk(scientific_versions)

if INSTALL_DEPENDENCIES:
    constraint_path = ROOT / "cache" / "runtime-scientific-constraints.txt"
    constraint_path.parent.mkdir(parents=True, exist_ok=True)
    constraint_path.write_text(
        "".join(
            f"{name}=={version}\n"
            for name, version in sorted(scientific_versions.items())
        ),
        encoding="utf-8",
    )
    install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
    ]
    if scientific_versions:
        install_command.extend(["--constraint", str(constraint_path)])
    install_command.extend(
        ["-r", str(CODE_DIR / "requirements-notebook.txt")]
    )
    run_checked(install_command, "Cài dependency notebook")

    versions_after_install = installed_versions(SCIENTIFIC_DISTRIBUTIONS)
    changed_versions = {
        name: (version, versions_after_install.get(name))
        for name, version in scientific_versions.items()
        if versions_after_install.get(name) != version
    }
    if changed_versions:
        raise RuntimeError(
            "Pip đã thay scientific stack dù có constraint: "
            f"{changed_versions}. Hãy xóa runtime và báo lại log cài đặt."
        )
    ensure_kernel_matches_disk(versions_after_install)

    run_checked(
        [
            sys.executable,
            "-c",
            "import numpy, pandas, scipy; "
            "print(numpy.__version__, pandas.__version__, scipy.__version__)",
        ],
        "Kiểm tra ABI NumPy/Pandas/SciPy",
    )

    if RUN_HEAVY_METRICS:
        family_requirements = {
            "BERTScore": "requirements-metric-bertscore.txt",
            "COMET": "requirements-metric-comet.txt",
            "BLEURT": "requirements-metric-bleurt.txt",
        }
        family_probes = {
            "BERTScore": "import bert_score, transformers",
            "COMET": "import lightning_utilities, pytorch_lightning, comet",
            "BLEURT": "import bleurt, tf_slim, sentencepiece, tensorflow",
        }
        for family, requirement in family_requirements.items():
            requirement_path = CODE_DIR / requirement
            package_dir = ROOT / "cache" / "metric-packages" / family.lower()
            fingerprint = hashlib.sha256(
                requirement_path.read_bytes()
                + f"{sys.version_info.major}.{sys.version_info.minor}".encode()
            ).hexdigest()[:16]
            ready_marker = package_dir / f".ready-{fingerprint}"

            probe_env = os.environ.copy()
            probe_env["PYTHONPATH"] = os.pathsep.join(
                [str(package_dir), str(CODE_DIR / "src")]
            )
            probe = subprocess.run(
                [sys.executable, "-c", family_probes[family]],
                env=probe_env,
                text=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
            )
            if not ready_marker.is_file() or probe.returncode != 0:
                # Cache cài dở được bỏ riêng theo family, không ảnh hưởng model cache.
                if package_dir.is_dir():
                    shutil.rmtree(package_dir)
                package_dir.mkdir(parents=True, exist_ok=True)
                print(f"Cài package tối thiểu cho metric: {family}")
                run_checked(
                    [
                        sys.executable,
                        "-m",
                        "pip",
                        "install",
                        "-q",
                        "--upgrade",
                        "--no-deps",
                        "--target",
                        str(package_dir),
                        "-r",
                        str(requirement_path),
                    ],
                    f"Cài dependency {family}",
                )
                probe_env["PYTHONPATH"] = os.pathsep.join(
                    [str(package_dir), str(CODE_DIR / "src")]
                )
                run_checked(
                    [sys.executable, "-c", family_probes[family]],
                    f"Kiểm tra dependency {family}",
                    env=probe_env,
                )
                ready_marker.write_text("ready\n", encoding="utf-8")
            HEAVY_WORKER_PATHS[family] = package_dir

sys.path.insert(0, str(CODE_DIR / "src"))

import pandas as pd
from IPython.display import display
from vn_meta_judge.notebook_workflow import NotebookExperiment, parallel_jobs

experiment = NotebookExperiment(
    root=ROOT,
    code_dir=CODE_DIR,
    run_name=RUN_NAME,
    model=GEMINI_MODEL,
    limit=LIMIT,
    seed=SEED,
    api_keys=STANDARD_GEMINI_API_KEYS,
    high_quota_api_keys=HIGH_QUOTA_GEMINI_API_KEYS,
    api_workers=STANDARD_API_WORKERS,
    api_min_interval_seconds=STANDARD_API_MIN_INTERVAL_SECONDS,
    high_quota_workers_per_key=HIGH_QUOTA_WORKERS_PER_KEY,
    high_quota_min_interval_seconds=HIGH_QUOTA_MIN_INTERVAL_SECONDS,
    cpu_workers=CPU_WORKERS,
    worker_python=METRIC_PYTHON,
    heavy_worker_pythonpaths=HEAVY_WORKER_PATHS,
    heavy=RUN_HEAVY_METRICS,
    underthesea=RUN_UNDERTHESEA,
    max_tokens=MAX_OUTPUT_TOKENS,
)

print("Output:", experiment.output)
print("Git workspace:", ROOT)
print(
    "Gemini keys:",
    f"standard={len(STANDARD_GEMINI_API_KEYS)},",
    f"high_quota={len(HIGH_QUOTA_GEMINI_API_KEYS)},",
    f"request_workers={experiment.api_workers}",
)
print(
    "Chế độ dữ liệu:",
    f"translations={'regenerate' if REGENERATE_TRANSLATIONS else 'repo'},",
    f"damage={'regenerate' if REGENERATE_DAMAGE else 'repo'}",
)

## 2. Chuẩn bị dữ liệu và sinh bản dịch

Kế thừa notebook gốc: VLSP2022 vi–zh, chọn 300 câu với seed 42, Gemini
3.5 Flash Lite cho hệ A, Google Translate cho hệ B và ghép kết quả theo STT.
Mặc định dùng dữ liệu đã lưu. Bật `REGENERATE_TRANSLATIONS` để chạy lại các cell
lấy mẫu/sinh bản dịch; checkpoint giữ nguyên các câu đã dịch thành công.

In [ ]:
if REGENERATE_TRANSLATIONS:
    if LOCAL_SOURCE_VI and LOCAL_SOURCE_ZH:
        vi_path, zh_path = Path(LOCAL_SOURCE_VI), Path(LOCAL_SOURCE_ZH)
    else:
        from concurrent.futures import ThreadPoolExecutor
        from huggingface_hub import hf_hub_download

        def download_source(language):
            return hf_hub_download(
                repo_id="VLSP2023-MT/ViBidirectionMT-Eval",
                filename=f"VLSP2022/Test/public test/test.vi-zh.2022.{language}",
                repo_type="dataset",
                token=HF_TOKEN or None,
            )

        with ThreadPoolExecutor(max_workers=2) as pool:
            vi_path, zh_path = pool.map(download_source, ["vi", "zh"])
else:
    print("Dùng dữ liệu dịch đã lưu.")

In [ ]:
if REGENERATE_TRANSLATIONS:
    import random

    with open(vi_path, encoding="utf-8") as handle:
        vi_lines = [line.strip() for line in handle if line.strip()]
    with open(zh_path, encoding="utf-8") as handle:
        zh_lines = [line.strip() for line in handle if line.strip()]

    assert len(vi_lines) == len(zh_lines), "Lệch số dòng nguồn và tham chiếu."
    idx = sorted(random.Random(SEED).sample(range(len(vi_lines)), EXPECTED_ROWS))

    df_source = pd.DataFrame(
        [
            {
                "STT": stt,
                "ID_cau_VLSP": f"vi-zh-2022-test-{i + 1:04d}",
                "Cau_nguon_ZH": zh_lines[i],
                "Cau_tham_chieu_VI": vi_lines[i],
            }
            for stt, i in enumerate(idx, start=1)
        ]
    )
    translation_dir = experiment.output / "translations"
    translation_dir.mkdir(parents=True, exist_ok=True)
    df_source.to_csv(
        translation_dir / "300_sample.csv", index=False, encoding="utf-8-sig"
    )
    display(df_source.head())

### Sinh hệ A/B và lưu checkpoint

Giữ system prompt và temperature = 0 của notebook gốc. Hai hệ dịch cùng
một câu độc lập; ghi checkpoint và ghép STT sau khi nhận kết quả. Dùng lại
file kết quả để giữ chính xác các bản dịch của lượt chạy đã được chấm điểm.

In [ ]:
if REGENERATE_TRANSLATIONS:
    from vn_meta_judge.translation_workflow import translate_dataset

    translated = translate_dataset(
        df_source,
        translation_dir,
        api_keys=ALL_GEMINI_API_KEYS,
        model=GEMINI_MODEL,
        request_caller=experiment.gemini_caller,
        source_workers=experiment.api_workers,
        target_workers=2,
    )
    ACTIVE_TRANSLATIONS_FILE = translation_dir / "translations_reusable.csv"
    ACTIVE_SOURCE_FILE = None
    # Điểm cũ thuộc hai bản dịch trong Git, không được gán cho A/B vừa sinh lại.
    ACTIVE_SCORES_FILE = None
    display(translated[["STT", "Ban_dich_He_A", "Ban_dich_He_B"]].head())
else:
    print("Dùng bản dịch A/B và điểm human đã chốt trong Git.")

### Đọc dữ liệu dịch đã lưu — cell debug độc lập

Cell này nhận XLSX có sheet `Cham_diem` hoặc CSV chứa nguồn, tham chiếu và
bản dịch A/B. Kiểm tra STT, số dòng, trường rỗng và sự khớp với bảng điểm.
Sau setup, bắt đầu từ đây để chạy sinh damage, metric và phân tích trên dữ liệu đã có.

In [ ]:
translations = experiment.load_translations(
    ACTIVE_TRANSLATIONS_FILE,
    source_file=ACTIVE_SOURCE_FILE,
    scores_file=ACTIVE_SCORES_FILE,
    expected_rows=EXPECTED_ROWS,
    translation_generation=(
        "regenerated_translations"
        if globals().get("REGENERATE_TRANSLATIONS", False)
        else "reused_translations"
    ),
)

display(
    translations[
        [
            "STT",
            "Nguon_ZH",
            "Tham_chieu_VI",
            "Ban_dich_He_A",
            "Ban_dich_He_B",
        ]
    ].head()
)

print(f"Đã đọc {len(translations)} câu và {2 * len(translations)} bản dịch A/B.")
same_count = translations["Ban_dich_He_A"].eq(translations["Ban_dich_He_B"]).sum()
print(f"Hai hệ dịch giống nhau: {same_count}/{len(translations)} câu.")

## 3. Chuẩn bị dữ liệu và khảo sát

Lưu input của lượt chạy, tách tập human có điểm hợp lệ và chọn tham chiếu
cho thí nghiệm. Các dòng có cờ lỗi vẫn được giữ trong bản xuất đầy đủ để phân tích.

In [ ]:
input_summary = experiment.prepare()
display(pd.DataFrame([input_summary]))

lengths = pd.DataFrame(
    {
        "Source characters": translations["Nguon_ZH"].str.len(),
        "Reference syllables": translations["Tham_chieu_VI"].str.split().str.len(),
    }
)
display(lengths.describe().round(2))

## 4. Sinh dữ liệu suy giảm ngữ nghĩa

Run All mặc định dùng hai file damage đã chốt trong Git, kiểm tra lại schema và
sự khớp reference rồi tạo B1 theo luật. Đặt `REGENERATE_DAMAGE = True` khi cần
sinh một run mới; checkpoint vẫn lưu theo câu, prompt và mức damage.
Cache metric nhẹ được chuẩn bị song song với phần kiểm tra dữ liệu.

In [ ]:
if REGENERATE_DAMAGE:
    # Nếu RUN_NAME này trước đó chỉ copy artifact Git, bỏ đúng bản copy đó để
    # tương thích bản notebook cũ. Checkpoint đã sinh khác nội dung vẫn được resume.
    for branch, source_path in REPO_DAMAGE_FILES.items():
        target_path = experiment.generation_dir / f"{branch}.jsonl"
        if (
            target_path.is_file()
            and source_path.is_file()
            and hashlib.sha256(target_path.read_bytes()).digest()
            == hashlib.sha256(source_path.read_bytes()).digest()
        ):
            target_path.unlink()

initial_jobs = parallel_jobs(
    {
        "damage": lambda: experiment.generate(
            regenerate_damage=REGENERATE_DAMAGE,
            damage_files=(None if REGENERATE_DAMAGE else REPO_DAMAGE_FILES),
        ),
        "metric_cache": experiment.prepare_metric_cache,
    },
    workers=2,
)

generation = initial_jobs["damage"]
display(
    pd.DataFrame(
        [
            {
                "branch": name,
                **{
                    k: result.get(k)
                    for k in ["status", "rows", "api_error_rows", "reason"]
                },
            }
            for name, result in generation.items()
            if isinstance(result, dict)
        ]
    )
)
print("Metric cache:", initial_jobs["metric_cache"]["status"])

expected_generation = ["rule_based", "zero_shot", "few_shot"]
generation_not_ready = [
    name for name in expected_generation
    if generation.get(name, {}).get("status") != "ready"
]
if generation_not_ready:
    raise RuntimeError(
        "Generation chưa hoàn tất: "
        + ", ".join(generation_not_ready)
        + ". Kiểm tra resource hoặc giữ nguyên RUN_NAME để resume checkpoint."
    )

## 5. Chấm điểm và kiểm định metric

Human, B1, zero-shot và few-shot được ghép bằng offset có kiểm tra hash. Mỗi
cấu hình metric nặng chỉ nạp model một lần rồi score toàn bộ dataset; metric
nhẹ chạy song song theo family. Output sau đó được tách về đúng thứ tự nguồn.

In [ ]:
# Smoke test dùng một câu mỗi nhánh và một config đại diện cho từng họ metric.
if RUN_METRIC_SMOKE_TEST:
    smoke_result = experiment.metric_smoke_test(gpu_batch_size=4)
    display(
        pd.DataFrame(
            [
                {
                    "tokenization": name,
                    "status": item.get("status"),
                    "metrics": item.get("completed_metrics"),
                    "rows": item.get("combined_rows"),
                }
                for name, item in smoke_result["summaries"].items()
            ]
        )
    )
    if smoke_result["status"] != "ready":
        failed_logs = [
            item.get("log")
            for item in smoke_result["tasks"].values()
            if item.get("status") != "ready"
        ]
        raise RuntimeError(f"Smoke test metric thất bại. Log: {failed_logs}")
else:
    print("Bỏ smoke test theo cấu hình.")

In [ ]:
if RUN_FULL_METRICS:
    metric_result = experiment.score_fast(gpu_batch_size=GPU_BATCH_SIZE)
    display(
        pd.DataFrame(
            [
                {
                    "tokenization": name,
                    "status": item.get("status"),
                    "metrics": item.get("completed_metrics"),
                    "rows": item.get("combined_rows"),
                }
                for name, item in metric_result["summaries"].items()
            ]
        )
    )
    if metric_result["status"] != "ready":
        failed_logs = [
            item.get("log")
            for item in metric_result["tasks"].values()
            if item.get("status") != "ready"
        ]
        raise RuntimeError(f"Full metric chưa hoàn tất. Log: {failed_logs}")
else:
    metric_result = {
        "status": "skipped",
        "reason": "RUN_FULL_METRICS=False",
    }
    print(
        "Chế độ smoke-only: đã bỏ qua chấm full. "
        "Đặt RUN_FULL_METRICS=True sau khi smoke test chạy ổn."
    )

FULL_METRICS_READY = metric_result.get("status") == "ready"

### Tương quan với human judgment và mức damage

Sau khi mọi lượt chấm điểm kết thúc, tính `r_hum = corr(metric, human)`,
`r_syn = corr(metric, −damage)` và `MC = corr(r_hum, r_syn)`.
Bảng ghi số metric chung; các cấu hình có tương quan không xác định được ghi riêng.

In [ ]:
if FULL_METRICS_READY:
    correlation_table = experiment.correlate()
    display(correlation_table.round(4))
else:
    correlation_table = pd.DataFrame()
    print("Bỏ qua correlation vì chưa chạy full metric.")

## 6. Đối chiếu baseline và xuất điểm

B1 sử dụng nhiễu theo luật và được chấm cùng bộ metric với Gemini. B2 tính lại
meta-correlation từ artifact của tác giả. B2 và xuất điểm người chấm chạy song song.

In [ ]:
if FULL_METRICS_READY:
    analysis_results = experiment.analyze(manual_audit=MANUAL_AUDIT_FILE)
    display(
        pd.DataFrame(
            [
                {
                    "task": name,
                    "status": item["status"],
                    "rows": item.get("rows"),
                }
                for name, item in analysis_results.items()
            ]
        )
    )
else:
    analysis_results = {}
    print("Bỏ qua baseline và xuất điểm vì chưa chạy full metric.")

## 7. Phân tích lỗi

Tổng hợp lỗi generation, metric và thay đổi điểm theo mức damage. Mẫu audit
được xuất để người chấm ghi `observed_level`; kết quả audit chỉ tính trên nhãn đã cung cấp.

In [ ]:
if FULL_METRICS_READY:
    diagnostics = experiment.metric_diagnostics()
    display(diagnostics.head(12))

    audit_sample = experiment.manual_audit_sample(count=30)
    print("Mẫu kiểm tra mức damage:", audit_sample)
    print("Báo cáo lỗi:", experiment.analysis_dir / "error_analysis.json")
else:
    diagnostics = pd.DataFrame()
    audit_sample = None
    print("Bỏ qua phân tích lỗi vì chưa chạy full metric.")

## 8. Demo trên câu mới

Áp dụng B1 cho một câu ngoài benchmark, so sánh điểm BLEU/chrF qua sáu mức
damage. Demo minh họa độ nhạy metric; kết quả Gemini được báo cáo ở thí nghiệm chính.

In [ ]:
if FULL_METRICS_READY:
    demo_table = experiment.demo(DEMO_SENTENCE)
    display(demo_table)

    import matplotlib.pyplot as plt

    metric_columns = [c for c in demo_table.columns if c not in {"level", "text"}]
    if metric_columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        for column in metric_columns:
            ax.plot(
                demo_table["level"],
                demo_table[column],
                marker="o",
                label=column,
            )

        ax.set(xlabel="Damage level", ylabel="Metric score", xticks=range(6))
        ax.grid(alpha=0.2)
        ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        fig.savefig(
            experiment.analysis_dir / "demo.png",
            dpi=160,
            bbox_inches="tight",
        )
        display(fig)
        plt.close(fig)
else:
    demo_table = pd.DataFrame()
    print("Bỏ qua demo vì chưa chạy full metric.")

In [ ]:
if FULL_METRICS_READY:
    manifest = experiment.finalize()
    print("Trạng thái:", manifest["status"])
    print("Danh mục kết quả:", experiment.output / "manifest.json")
    display(pd.DataFrame(manifest["issues"]))
else:
    smoke_status = (
        smoke_result.get("status")
        if RUN_METRIC_SMOKE_TEST
        else "skipped"
    )
    manifest = {
        "status": "smoke-only",
        "smoke_status": smoke_status,
        "issues": [],
    }
    print("Trạng thái: smoke-only | metric smoke:", smoke_status)